# 📊 Exploratory Data Analysis — Telco Customer Churn

> **Objective:** Understand dataset structure, feature distributions, and key churn drivers before building the ML pipeline.

---

## 1. Imports & Setup

In [ ]:
import os, sys, warnings
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.ticker as mtick
import seaborn as sns

warnings.filterwarnings("ignore")
sns.set_theme(style="whitegrid", palette="muted", font_scale=1.1)
plt.rcParams["figure.dpi"] = 110

NOTEBOOK_DIR = os.getcwd()
DATA_PATH    = os.path.join(NOTEBOOK_DIR, "..", "data", "telco_churn.csv")
FIG_DIR      = os.path.join(NOTEBOOK_DIR, "figures")
os.makedirs(FIG_DIR, exist_ok=True)
print("Setup complete ✓")

## 2. Load Dataset

In [ ]:
df = pd.read_csv(DATA_PATH)
print(f"Shape: {df.shape}")
df.head()

## 3. Dataset Overview

| Attribute | Value |
|-----------|-------|
| **Rows** | 7,043 customers |
| **Columns** | 21 (demographics, services, billing, churn) |
| **Target** |  — Yes / No |

Each row represents one customer. Features cover account demographics, subscribed services, billing info, and whether the customer churned.

In [ ]:
print("── Data Types ──")
print(df.dtypes)
print()
print("── Statistical Summary ──")
df.describe()

## 4. Missing Values & Cleaning

In [ ]:
print("Missing values per column:")
print(df.isnull().sum())
print()
# TotalCharges is object due to whitespace entries
df["TotalCharges"] = pd.to_numeric(df["TotalCharges"], errors="coerce")
missing = df["TotalCharges"].isna().sum()
print(f"TotalCharges whitespace entries (now NaN): {missing}")
df["TotalCharges"].fillna(df["TotalCharges"].median(), inplace=True)
# Numeric target
df["ChurnNum"] = df["Churn"].map({"Yes": 1, "No": 0})
print("Cleaned successfully ✓")

## 5. Churn Distribution

> **Insight:** ~26% of customers churned — a moderate class imbalance. We apply  in models to handle this.

In [ ]:
from IPython.display import Image
Image(filename=os.path.join(FIG_DIR, "churn_distribution.png"))

In [ ]:
churn_counts = df["Churn"].value_counts()
print(churn_counts)
print(f"
Churn rate: {churn_counts["Yes"]/len(df)*100:.1f}%")

## 6. Numeric Feature Distributions by Churn

> **Insights:**
> - **Tenure**: churners concentrate heavily in 0–12 months; loyal customers persist past 48 months.
> - **Monthly Charges**: churners skew toward higher charges (0–00).
> - **Total Charges**: non-churners dominate high totals, reflecting long tenure.

In [ ]:
Image(filename=os.path.join(FIG_DIR, "numeric_distributions.png"))

In [ ]:
print("Average tenure by churn:")
print(df.groupby("Churn")["tenure"].mean().round(1))
print("
Average monthly charges by churn:")
print(df.groupby("Churn")["MonthlyCharges"].mean().round(2))

## 7. Churn by Contract Type

> **Key Insight:** Month-to-month customers churn at ~43% — nearly **9× higher** than two-year contract holders (~3%). Contract type is the strongest categorical predictor.

In [ ]:
Image(filename=os.path.join(FIG_DIR, "churn_by_contract.png"))

In [ ]:
contract_churn = df.groupby("Contract")["ChurnNum"].mean().sort_values(ascending=False)*100
print("Churn rate by contract type:")
print(contract_churn.round(1))

## 8. Churn by Tenure

> **Insight:** Churn probability is ~50% in the first year and drops to under 10% after 4 years. Early-stage customer retention programs are the highest leverage intervention.

In [ ]:
Image(filename=os.path.join(FIG_DIR, "churn_by_tenure.png"))

## 9. Churn by Monthly Charges

> **Insight:** Customers paying 6+ monthly churn at 35%+ — likely feeling over-charged relative to perceived value. Bundled plan offers could reduce this.

In [ ]:
Image(filename=os.path.join(FIG_DIR, "churn_by_charges.png"))

## 10. Churn by Payment Method

> **Insight:** Electronic check payers churn at ~45% — almost double mailed check users (~19%). Auto-pay enrollment may be a proxy for commitment and financial stability.

In [ ]:
Image(filename=os.path.join(FIG_DIR, "churn_by_payment.png"))

## 11. Correlation Analysis

> **Insights:**
> -  has the strongest **negative** correlation with churn (−0.35): longer customers are safer.
> -  has a **positive** correlation (+0.19): higher bills → higher risk.
> -  correlates negatively because it is a function of long tenure.

In [ ]:
Image(filename=os.path.join(FIG_DIR, "correlation_heatmap.png"))

In [ ]:
numeric_df = df.select_dtypes(include="number").drop(columns=["ChurnNum"], errors="ignore")
numeric_df["Churn"] = df["ChurnNum"]
print("Correlations with Churn:")
print(numeric_df.corr()["Churn"].sort_values(ascending=False))

## 12. Key Insights Summary

| # | Finding | Business Implication |
|---|---------|---------------------|
| 1 | **Month-to-month** contract → 43% churn rate | Incentivise upgrades to 1-year or 2-year plans |
| 2 | **New customers** (0–12 mo) churn at ~50% | Implement 90-day onboarding loyalty programmes |
| 3 | **Electronic check** → ~45% churn | Promote auto-pay enrollment with a discount |
| 4 | **Fiber optic** users churn more than DSL | Audit service quality; consider SLA guarantees |
| 5 | **High monthly charges** (6+) drive churn | Bundle services to increase perceived value |
| 6 |  is the strongest single predictor | Long-term retention is self-reinforcing |

---

**Next Step →**  uses these insights to build and evaluate ML classifiers.